# Clustering
Supervised machine learning is maybe the most known branch of AI because it can solve many kind of tasks. On the other hand unsupervised approaches works well for text, audio or image generation also leving aside generative models, there other approaches there are interesting tasks like outliner detection, dimensional reduction or clustering. This time we're going to talk about **clustering**, maybe one of the most known tasks of unsupervised learning.

## Definition
Clustering approaches are a family of unsupervised machine learning algorithms used to group similar data points together. It helps discover hidden patterns or natural groupings in datasets by placing similar data points into the same cluster. Each data point is assigned to a cluster based on similarity or distance measures. A clustering task can be established in 2 ways:

- **Hard clustering:** Assigns each sample to eaxctly one cluster. A sample can't belongs to multiple cluster avoiding overlapping. This approach is easy to interpret.
- **Soft clustering:** Allows a data sample to belong to multiple clusters with different confidence measures. It estimates a probability of belonging to each cluster.

## Types of clustering 

### Centroid based clustering
Groups data points around central points called centroids. Each cluster is represented by a central point, and data points are assigned to the nearest center. This strategy is fast and scalable for large dataset and simple to implement and interpret; however, is sentitive to initialization values and outliners also requires manual selection of the amount of clusters.

<center>
    <img width="200px" height="200px"  src="https://developers.google.com/machine-learning/clustering/images/CentroidBasedClustering.svg" />
</center>

### Density based clustering
Identifies clusters as regions where data points are densely packed together. This approach can find any amount of clusters of any shape, working fine for noisy datasets but having slow downgrades on high dimensional datasets. Points in low density areas (outliners) are treated as noise and a cluster isn't assigned.

<center>
    <img width="200px" height="200px" src="https://developers.google.com/static/machine-learning/clustering/images/DensityClustering.svg?hl=es-419" />
</center>

### Hieralchical clustering
Constructs a tree of clusters by gradually merging or splitting groups of data points that shows relationships between clusters. The root of the tree is the unique cluster that gathers all the samples, the leaves being the clusters with only one sample. It's computationally intensive for large datasets and the splittinf decisions are irreversible but it's easy to visualize.

<center>
    <img width="200px" height="200px" src="https://developers.google.com/machine-learning/clustering/images/HierarchicalClustering.svg" />
</center>

### Distribution-based clustering
Assumes that data points come from a mixture of probability distributions. Each cluster is modelled as a statistical distribution this method is more computationally expensive than the approaches presented previously and is sensitive to initialization. Works well for clusters that aren't perfectly circular and is suitable for overlapping clusters.

<center>
    <img width="200px" height="200px" src="https://developers.google.com/machine-learning/clustering/images/DistributionClustering.svg" />
</center>

### Fuzzy clustering
Allows data points to belong to multiple clusters with different degrees of membership, beign useful when cluster boundaries aren't clear. It's perfect for complex or imprecise data and models data ambiguity explicitly.

<center>
    <img width="400px" height="200px" src="https://media.geeksforgeeks.org/wp-content/uploads/20250911173529200884/Fuzzy--768.webp" />
</center>

## Algorithms
### K-means
A centroid based clustering approach that iteratively assigns points to nearest centroid and recalculates centroids to minimize intra cluster variance. It computes the euclidean distance between 2 data points to determine how far are from the others, is called "K-means" because $k$ is indicates the number of clusters to be computed and means as the way as the centroids are calculated. This approach assumes that the clusters are compact and spherical.

#### Algorithm
1. Select $k$ cluster centroids randomly setting random values to $c_j$.
2. Assign each data point to the nearest centroid $C_j$, calculating the distance measure $d(x_i, c_j)$ for each cluster $c_j$ and instance $x_i$.

$$
    C_j = \{ x_i | \forall_{i} \forall{c_k} (j \not{=} k) \rightarrow (d(x_i, c_j) \leq d(x_i, c_k)) \}
$$

3. Recalculate the centroid $c_j$ of each cluster by averaging the point within it.
$$
    c_j = \frac{1}{|C_j|} \sum_{x_i \in C_j} x_i
$$
3. Repeat from step 1 to 3 until the cetroinds no longer change or the maximum number of iterations is reached.

#### Implementation
Scikit-learn library has the `KMeans` class and has and alternative implementation, faster than the default that implements mini-batching approach at `MiniBatchKMeans` class.

- **`n_clusters`:** The number of clusters to form as well as the number of centroids to generate.
- **`init`:** Initial centroids initialization method.
- **`max_iter`:** Maximum number of iterations of the algorithm for a single run.
- **`algorithm`:** K-means algorithm to use.
- **`tol`:** Relative tolerance with regards to Frobenius norm of the difference in the cluster centers of two consecutive iterations to declare convergence.
- **`n_init`:** Number of times the K-means algorithm is run with different centroid seeds. The final results is the best seed of consecutive runs. Establishes the initial value for the centroids.
- **`batch_size`:** Size of the mini batches, only available at `MiniBatchKMeans`.
- **`max_no_improvement`:** Control early stopping based on the consecutive number of mini batches that doesn't yield an improvement on the smoothed inertia.

In [ ]:
import sklearn.cluster import KMeans
from pandas import read_csv
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

PATH = "/kaggle/input/datasets/waddahali/fraud-detection/fraud.csv"

DATA = read_csv(PATH)

NUMS = ["transaction_amount", "num_items", "customer_age", "prev_transactions", "distance_from_home", "network_quality", "velocity_score"]
CATS = ["hour_of_day", "is_weekend", "device_type", "is_first_transaction", "store_type", "is_fraud"]

PIPELINE_NUMS = Pipeline(steps=[
    ("fill_na", SimpleImputer(strategy="mean")),
    ("standarize", StandardScaler())
])

PIPELINE_CATS = Pipeline(steps=[
    ("fill_na", SimpleImputer(strategy="most_frequent")),
])

TRANSFORMER = ColumnTransformer([
    ("nums", PIPELINE_NUMS, NUMS),
    ("cats", PIPELINE_CATS, CATS)
])


NORMLIZED_DATA = TRANSFORMER.fit_transform(DATA)

KMEANS = KMeans(random_state=123, n_clusters=6)

KMEANS.fit_predict(NORMALIZED_DATA)

### DBSCAN
A density-based clustering algorithm that groups data points that are closely packed together and marks outliners as noise based on their density in the feature space. Clusters are defined as dense regions in the data space separated by areas of lower density. Performs well for non spherical or convex cluster shpes, and has the advantage of non being affected by outliners.

#### Definitions
- **Core points:** Data points which have a sufficient number of neighbors within a specified radius $\epsilon$.
- **Border points:** Points near to core points but lack enough neigbors to be core points themselves.
- **Noise points:** Points that don't belong to any cluster, usually are outliners.

#### Algorithm
1. For each point in the dataset, count the number of points within its $\epsilon$-neighborhood. If the count meets or exceeds a threshold $\lambda$, mark the point as a core point.
2. For each core point that isn't alredy assigned to a cluster creater a new cluster. Recursively, find all density-connected points within the $\epsilon$ radius of the core point and add them to the cluster.
3. After processing all points any point that doesn't belong to a cluster is labeled as noise.

#### Implementation
Scikit library has the `DBSCAN` class with the following hyperparameters:

- **`eps`:** $\epsilon$-distance between to samples for one to be considered as in the neighborhood of the other.
- **`min_samples`:** The number of samples in a neighborhood for a point to be considered as a core point.
- **`metric`:** The metric to use when calculating distance between instance in a feature array.
- **`algorithm`:** The algorithm to be used to compute pointwise distances and find nearest neighbors.
- **`p`:** Power of the Minkowski metric to be used to calculate distance between points.

In [ ]:
from sklearn.cluster import DBSCAN

DBSCAN_MODEL = DBSCAN(eps=1, min_samples=10, metric="euclidean")

DBSCAN_MODEL.fit_predict(DATA)

### OPTICS
An extension of DBSCAN algorithm that address the high varying densities issue that requires a higher $\epsilon$ radius for sparser regions and smaller for high density areas

#### Core concepts
The concepts of 
- **Core distance:** The minimum distance ($\epsilon$ radius) needed for a point to be classified as a core point.
- **Reachability distance:** It's a measure of how difficult it is to reach from one point to another. It's calculated as the larger core distance of the starting point and the actual point.

<center>
    <img width="400px" height="200px" src="https://media.geeksforgeeks.org/wp-content/uploads/20250912153243861514/OpticCLuster-768.webp" />
</center>

#### Algorithm
1. Select a starting point and checks if it has at least $\lambda$ neighbors within $\epsilon$-radius. If does, it is marked as a core point and nearby points are then analyzed.
2. Reachability distance is computed for each neighboring point.
3. Points are then processed in order of their reachability distance hence forming a reachability plot.
4. Clusters appears as low reachability distances and noise as high.

#### Implementation
Scikit library has the `OPTICS` class with the following hyperparameters:

- **`min_samples`:** The number of samples in a neighborhood for a point to be considered as a core point.
- **`max_eps`:** The maximum distance between two samples for one to be considered as in the neighborhood of the other.
- **`metric`:** The metric to use when calculating distance between instance in a feature array.
- **`algorithm`:** The algorithm to be used to compute pointwise distances and find nearest neighbors.
- **`p`:** Power of the Minkowski metric to be used to calculate distance between points.
- **`cluster_method`:** The extraction method used to extract clusters using the calculated reachability and ordering.
- **`eps`:** $\epsilon$-distance between to samples for one to be considered as in the neighborhood of the other.

In [ ]:
from sklearn.cluster import OPTICS

OPTICS_MODEL = OPTICS(eps=1, min_samples=10, metric="euclidean", cluster_method="xi", xi=0.2)

OPTICS_MODEL.fit_predict(DATA)

### Agglomerative
A hieralchical clustering approach to group similar data points into clusters based on their proximity. It follows a bottom-up approach, where each data point starts as its own cluster and gradually merges with other based on similarity. The merging continues until all points form a single cluster or a set number of clusters remain.

#### Linkage criteria
- **Ward:** Minimizes the sum of squared differences within all clusters. It is a variance-minimizing approach and in this sense is similar to the k-means objective function but tackled with an agglomerative hierarchical approach.
- **Maximum linkage:** Minimizes the maximum distance between observations of pairs of clusters.
- **Average linkage:** Minimizes the average of the distances between all observations of pairs of clusters.
- **Single linkage:** Minimizes the distance between the closest observations of pairs of clusters.

#### Algorithm
1. Treat each data point as its own cluster, at begining you will have as clusters as data points.
2. Compute a distance matrix between every pair of clusters.
3. Identify the two clusters that are closest based on the chosen linkage method and combine them.
4. Recalculate the distance between the newly formed cluster and all remaining clusters.
5. Repeat steps 3 and 4 until a distance threshold or a predefined number of clusters is reached.

#### Implementation
Scikit has the `AgglomerativeClustering` class with the following hyperparameters:

- **`n_clusters`:** The number of clusters to find.
- **`metric`:** The metric to use when calculating distance between instance in a feature array.
- **`compute_full_tree`:** When set to `True`, stops early construction of the tree at `n_clusters`.
- **`linkage`:** Linkage criterion to use.
- **`distance_threshold`:** The linkage distance threshold at or above which clusters will not be merged.
- **`compute_distances`:** Computes distances between clusters.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

HIERALCHICAL = AgglomerativeClustering(n_clusters=8, metric="euclidean", linkage="ward")

HIERALCHICAL.fit_predict(DATA)